In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report

df = pd.read_csv('uci_heart_disease_dataset.csv')
X = df.drop(columns=['target'])
y = df['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 3. Scale variables
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 4. Initialize learning models
models = {
    "Logistic Regression": LogisticRegression(random_state=42),
    "Support Vector Machine": SVC(kernel='rbf', probability=True, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "XGBoost": XGBClassifier(eval_metric='logloss', random_state=42)
}

# 5. Execute evaluation loops
accuracies = {}
print("==========================================================")
print("              DISEASE PREDICTION MODEL SCORES             ")
print("==========================================================\n")

for name, model in models.items():
    if name in ["Logistic Regression", "Support Vector Machine"]:
        model.fit(X_train_scaled, y_train)
        predictions = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        predictions = model.predict(X_test)
        
    acc = accuracy_score(y_test, predictions)
    accuracies[name] = acc
    
    print(f"--- {name} ---")
    print(f"Accuracy Score: {acc * 100:.2f}%")
    print("\nClassification Report:")
    print(classification_report(y_test, predictions, target_names=["No Disease", "Disease"]))
    print("-" * 58)

# 6. Final Comparative Performance Ranking
sorted_rankings = sorted(accuracies.items(), key=lambda x: x[1], reverse=True)
print("\n==========================================================")
print("               FINAL ALGORITHM RANKINGS                   ")
print("==========================================================")
for rank, (model_name, score) in enumerate(sorted_rankings, 1):
    print(f"Rank {rank}: {model_name.ljust(25)} -> Accuracy: {score * 100:.2f}%")
print("==========================================================")

              DISEASE PREDICTION MODEL SCORES             

--- Logistic Regression ---
Accuracy Score: 80.00%

Classification Report:
              precision    recall  f1-score   support

  No Disease       0.79      0.82      0.80        60
     Disease       0.81      0.78      0.80        60

    accuracy                           0.80       120
   macro avg       0.80      0.80      0.80       120
weighted avg       0.80      0.80      0.80       120

----------------------------------------------------------
--- Support Vector Machine ---
Accuracy Score: 76.67%

Classification Report:
              precision    recall  f1-score   support

  No Disease       0.76      0.78      0.77        60
     Disease       0.78      0.75      0.76        60

    accuracy                           0.77       120
   macro avg       0.77      0.77      0.77       120
weighted avg       0.77      0.77      0.77       120

----------------------------------------------------------
--- Random Fore

In [2]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# 1. Create interactive widgets for every feature
form_items = [
    widgets.IntSlider(name='age', value=50, min=1, max=100, description='Age:'),
    widgets.Dropdown(name='sex', options=[('Male', 1), ('Female', 0)], description='Sex:'),
    widgets.Dropdown(name='cp', options=[('Typical Angina', 0), ('Atypical Angina', 1), ('Non-anginal Pain', 2), ('Asymptomatic', 3)], description='Chest Pain:'),
    widgets.IntSlider(name='trestbps', value=120, min=80, max=200, description='Resting BP:'),
    widgets.IntSlider(name='chol', value=200, min=100, max=600, description='Cholesterol:'),
    widgets.Dropdown(name='fbs', options=[('> 120 mg/dl', 1), ('<= 120 mg/dl', 0)], description='Fasting BS:'),
    widgets.Dropdown(name='restecg', options=[('Normal', 0), ('ST-T Wave Abnormality', 1), ('Left Ventricular Hypertrophy', 2)], description='Resting ECG:'),
    widgets.IntSlider(name='thalach', value=150, min=60, max=220, description='Max Heart Rate:'),
    widgets.Dropdown(name='exang', options=[('Yes', 1), ('No', 0)], description='Ex. Angina:'),
    widgets.FloatSlider(name='oldpeak', value=0.0, min=0.0, max=6.0, step=0.1, description='Oldpeak:'),
    widgets.Dropdown(name='slope', options=[('Upsloping', 0), ('Flat', 1), ('Downsloping', 2)], description='ST Slope:'),
    widgets.Dropdown(name='ca', options=[('0', 0), ('1', 1), ('2', 2), ('3', 3)], description='Major Vessels:'),
    widgets.Dropdown(name='thal', options=[('Normal', 1), ('Fixed Defect', 2), ('Reversable Defect', 3)], description='Thalassemia:')
]

# Create a container for the form inputs
ui = widgets.VBox([widgets.HBox([w]) for w in form_items])
predict_button = widgets.Button(description="Predict Diagnosis", button_style='success')
output = widgets.Output()

# 2. Define the prediction action logic
def on_button_clicked(b):
    with output:
        clear_output()
        
        # Extract current values from the widgets
        user_input_data = {w.description.replace(':', '').strip(): w.value for w in form_items}
        
        # Remap the UI descriptions back to original dataset column names
        mapping = {
            'Age': 'age', 'Sex': 'sex', 'Chest Pain': 'cp', 'Resting BP': 'trestbps',
            'Cholesterol': 'chol', 'Fasting BS': 'fbs', 'Resting ECG': 'restecg',
            'Max Heart Rate': 'thalach', 'Ex. Angina': 'exang', 'Oldpeak': 'oldpeak',
            'ST Slope': 'slope', 'Major Vessels': 'ca', 'Thalassemia': 'thal'
        }
        final_data = {mapping[k]: v for k, v in user_input_data.items()}
        
        # Convert to DataFrame
        user_df = pd.DataFrame([final_data])
        
        # Use XGBoost model
        best_model = models["XGBoost"]
        prediction = best_model.predict(user_df)
        prediction_proba = best_model.predict_proba(user_df)
        
        # Display nicely formatted outputs
        print("==========================================================")
        print("               DYNAMIC PATIENT DIAGNOSIS                  ")
        print("==========================================================")
        if prediction[0] == 1:
            print(f"Result: PREDICTED DISEASE")
            print(f"Confidence: {prediction_proba[0][1] * 100:.2f}% chance of heart disease.")
        else:
            print(f"Result: NO DISEASE DETECTED")
            print(f"Confidence: {prediction_proba[0][0] * 100:.2f}% chance of being healthy.")
        print("==========================================================")

predict_button.on_click(on_button_clicked)

# 3. Render the interactive GUI in your notebook
display(ui, predict_button, output)

Button(button_style='success', description='Predict Diagnosis', style=ButtonStyle())

Output()